# Embedding-Based Strategy — Ported from portifolio.ipynb

Loads cached embedding data, applies S1_hard70_mom3_rev (best performer), runs through `src/backtest/` engine.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
_root = Path.cwd()
if (_root / 'src').exists(): pass
elif (_root.parent / 'src').exists(): _root = _root.parent
elif (_root.parent.parent / 'src').exists(): _root = _root.parent.parent
sys.path.insert(0, str(_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.backtest.engine import decompose_alpha_beta
from src.backtest.visualization import BG, PANEL, GOLD, GREEN, RED, WHITE, CYAN

In [ ]:
cache_pkl = Path('cache/data/top50_prepared_data.pkl')
cached = pd.read_pickle(cache_pkl)

ret_df = cached['ret_df']
intensity_df = cached['intensity_df']
known_ret = cached['known_ret']
regime = cached['regime']
mask = cached['mask']
split = 2694

pd.DataFrame({
    '': ['ret_df', 'intensity_df', 'known_ret', 'split', 'tickers', 'dates'],
    'shape / value': [str(ret_df.shape), str(intensity_df.shape), str(known_ret.shape), split, ret_df.shape[1], ret_df.shape[0]]
})

## 1. Signal Construction — S1_hard70_mom3_rev

Exact formula from portifolio.ipynb (best performer):
```
int_rank = intensity.rank(axis=1, pct=True)
hard70 = (int_rank >= 0.70)
S1 = hard70 × (regime × reversal + (1-regime) × momentum_3)
```

In [ ]:
reg_mat = pd.DataFrame(
    np.repeat(regime.to_numpy()[:, None], ret_df.shape[1], axis=1),
    index=ret_df.index, columns=ret_df.columns,
)

int_rank = intensity_df.rank(axis=1, pct=True)
hard70 = (int_rank >= 0.70).astype(float)

rev1 = -np.sign(known_ret).fillna(0.0)
mom3 = np.sign(known_ret.rolling(3, min_periods=2).mean()).fillna(0.0)

raw_signal = hard70 * (reg_mat * rev1 + (1.0 - reg_mat) * mom3)

pd.DataFrame({
    'gate': ['hard70 (>= 70th pct)'],
    'routing': ['regime × reversal + (1-regime) × mom3'],
    'signal_nonzero_pct': [f'{(raw_signal.abs() > 0).mean().mean()*100:.1f}%'],
    'train_dates': [f'{raw_signal.index[0].date()} -> {raw_signal.index[split-1].date()}'],
    'test_dates': [f'{raw_signal.index[split].date()} -> {raw_signal.index[-1].date()}'],
})

## 2. Portfolio Construction

Tilt around equal-weight, clip long-only, normalize. Exposure overlay based on market trend.

In [ ]:
def build_tilt(raw_alpha):
    x = (raw_alpha * mask).fillna(0.0)
    x = x.sub(x.mean(axis=1), axis=0)
    x = x.div(x.abs().sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
    return x

tilt = build_tilt(raw_signal)

bh_w = mask.div(mask.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)
bh_ret = (bh_w * ret_df.fillna(0.0)).sum(axis=1)

mkt_trend = known_ret.mean(axis=1).rolling(20, min_periods=10).mean().fillna(0.0)

k = 0.05
up = 1.30
down = 1.00

exposure = pd.Series(np.where(mkt_trend > 0.0, up, down), index=ret_df.index)
w_tilted = (bh_w + k * tilt).clip(lower=0.0)
w_tilted = w_tilted.div(w_tilted.sum(axis=1).replace(0.0, np.nan), axis=0).fillna(0.0)

base_ret = (w_tilted * ret_df.fillna(0.0)).sum(axis=1)
strat_ret = exposure * base_ret

pd.DataFrame({
    '': ['k', 'up', 'down', 'train_ret', 'test_ret'],
    'value': [k, up, down,
              f'{(1+strat_ret.iloc[:split]).prod()-1:.2%}',
              f'{(1+strat_ret.iloc[split:]).prod()-1:.2%}']
})

## 3. Strategy vs Buy & Hold — Full History

In [ ]:
eq_strat = (1.0 + strat_ret).cumprod() * 100
eq_bh = (1.0 + bh_ret).cumprod() * 100

strat_train = strat_ret.iloc[:split]
strat_test = strat_ret.iloc[split:]
bh_train = bh_ret.iloc[:split]
bh_test = bh_ret.iloc[split:]

def stats(r):
    r = r.fillna(0.0)
    total = (1.0 + r).prod() - 1.0
    ann = (1.0 + total) ** (252.0 / max(len(r), 1)) - 1.0
    vol = r.std(ddof=0) * np.sqrt(252.0)
    sharpe = ann / vol if vol > 0 else 0.0
    dd = (r.cumsum().cummax() - r.cumsum()).max()
    return total, ann, vol, sharpe, dd

st_total, st_ann, st_vol, st_sharpe, st_dd = stats(strat_test)
bh_total, bh_ann, bh_vol, bh_sharpe, bh_dd = stats(bh_test)

fig, ax = plt.subplots(figsize=(20, 10))
fig.patch.set_facecolor(BG)

ax.plot(eq_strat.index, eq_strat.values, color=GOLD, lw=2.5, label='S1 Hard70 (embedding)')
ax.plot(eq_bh.index, eq_bh.values, color=WHITE, lw=1.2, ls='--', alpha=0.5, label='Buy & Hold (equal-weight)')
ax.axvline(eq_strat.index[split], color=WHITE, ls=':', alpha=0.3, label='Train/Test split')
ax.fill_between(eq_strat.index, eq_strat.values, 100, where=eq_strat.values >= 100, color=GREEN, alpha=0.06)
ax.fill_between(eq_strat.index, eq_strat.values, 100, where=eq_strat.values < 100, color=RED, alpha=0.06)
ax.axhline(100, color='white', ls='--', alpha=0.15)

ax.set_title(f'S1 Hard70 (Embedding) vs Buy & Hold — Top 50 US Stocks  |  Test: {st_total*100:+.1f}% vs BH {bh_total*100:+.1f}%  |  alpha={st_ann*100-bh_ann*100:+.1f}%  |  sharpe={st_sharpe:.2f}',
             color='white', fontsize=11, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')

fig.tight_layout()
fig.savefig(_root / 'images' / 'embedding_strategy_vs_bh.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 4. Alpha / Beta Decomposition

In [ ]:
ab = decompose_alpha_beta(strat_test, bh_test, 252)
pd.Series({
    'Alpha (annual %)': f"{ab['alpha_pct']:+.2f}%",
    'Beta': f"{ab['beta']:.3f}",
    'Beta Return (annual %)': f"{ab['beta_return_pct']:+.2f}%",
    'R-squared': f"{ab['r_squared']:.3f}",
}).to_frame('Value')

## 5. Drawdown Profile

In [ ]:
test_eq = eq_strat.iloc[split:]
test_bh = eq_bh.iloc[split:]
dd_strat = (test_eq / test_eq.cummax() - 1) * 100
dd_bh = (test_bh / test_bh.cummax() - 1) * 100

fig, ax = plt.subplots(figsize=(20, 7))
fig.patch.set_facecolor(BG)

ax.fill_between(dd_strat.index, dd_strat.values, 0, color=RED, alpha=0.15)
ax.plot(dd_strat.index, dd_strat.values, color=RED, lw=1.5, label=f'Strategy (max={dd_strat.min():.1f}%)')
ax.plot(dd_bh.index, dd_bh.values, color=WHITE, lw=1, ls='--', alpha=0.5, label=f'BH (max={dd_bh.min():.1f}%)')
ax.axhline(0, color='white', lw=0.5)

ax.set_title('Drawdown Profile — Test Set', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='lower left')

fig.tight_layout()
fig.savefig(_root / 'images' / 'embedding_strategy_drawdown.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 6. Rolling Alpha / Beta (63-day)

In [ ]:
window = 63
rolling_alpha = []
rolling_beta = []
dates_roll = []

for i in range(window, len(strat_test)):
    s = strat_test.iloc[i-window:i]
    b = bh_test.iloc[i-window:i]
    ab_w = decompose_alpha_beta(s.reset_index(drop=True), b.reset_index(drop=True), 252)
    rolling_alpha.append(ab_w['alpha_pct'])
    rolling_beta.append(ab_w['beta'])
    dates_roll.append(strat_test.index[i])

fig, ax = plt.subplots(figsize=(20, 8))
fig.patch.set_facecolor(BG)

ax.plot(dates_roll, rolling_alpha, color=GOLD, lw=1.5, label=f'Rolling Alpha (mean={np.mean(rolling_alpha):+.1f}%)')
ax.axhline(0, color='white', lw=0.5)
ax.set_ylabel('Alpha (%)', color='white', fontsize=10)
ax.set_title('Rolling CAPM Alpha (63-day window) — Test Set', color='white', fontsize=14, fontweight='bold')
ax.set_facecolor(PANEL)
ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12)
ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')

fig.tight_layout()
fig.savefig(_root / 'images' / 'embedding_strategy_rolling_alpha.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 7. Full Metrics Summary

In [ ]:
st_total, st_ann, st_vol, st_sharpe, st_dd = stats(strat_test)
bh_total_t, bh_ann_t, bh_vol_t, bh_sharpe_t, bh_dd_t = stats(bh_test)
ab = decompose_alpha_beta(strat_test, bh_test, 252)

pd.DataFrame({
    'Metric': [
        'Test Return', 'BH Return', 'Excess Return',
        'Ann. Return', 'BH Ann. Return', 'Ann. Alpha',
        'Ann. Volatility', 'BH Ann. Volatility',
        'Sharpe', 'BH Sharpe',
        'Max Drawdown', 'BH Max Drawdown',
        'CAPM Beta', 'CAPM Alpha (ann)', 'R-squared',
    ],
    'Value': [
        f'{st_total*100:+.2f}%', f'{bh_total_t*100:+.2f}%', f'{(st_total-bh_total_t)*100:+.2f}%',
        f'{st_ann*100:+.2f}%', f'{bh_ann_t*100:+.2f}%', f'{ab["alpha_pct"]:+.2f}%',
        f'{st_vol*100:.2f}%', f'{bh_vol_t*100:.2f}%',
        f'{st_sharpe:.3f}', f'{bh_sharpe_t:.3f}',
        f'{st_dd*100:.2f}%', f'{bh_dd_t*100:.2f}%',
        f'{ab["beta"]:.3f}', f'{ab["alpha_pct"]:+.2f}%', f'{ab["r_squared"]:.3f}',
    ]
})